In [99]:
from scipy.stats import norm
import biogeme.biogeme as bio
from biogeme.expressions import Beta, Variable, bioDraws, MonteCarlo, exp, log, Elem, bioNormalCdf
from biogeme import models
from biogeme.models import ordered_probit, ordered_logit
from biogeme import results as res
from biogeme.results import compile_estimation_results, calcPValue
import pandas as pd
import biogeme.database as db
import numpy as np
import biogeme.distributions as dist
import pickle
from urllib.request import urlopen
import os

In [100]:
df=pd.read_csv('final_dataset copie.csv')

/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_80224/1150838593.py:1: DtypeWarning: Columns (16,50,51,55,56,65,67,69,77,84,85,86,87,88,89,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('final_dataset copie.csv')


## Sanity check

In [101]:
df['Gender_driver'] = df['Gender_driver'].astype(str)
df['Helmet_driver'] = df['Helmet_driver'].astype(str)

In [102]:
df=df[df['severity']!=-1]

In [103]:
df=df.loc[df['Vehicle'].isin(['E-scooter','Bike','E-bike','Pedestrian'])]

In [104]:
# Remplacer les NaN par une nouvelle catégorie
df['Road width'] = df['Road width'].fillna('Missing')
df['age_2'] = df['age_2'].fillna(999)
df['Maneuver_2'] = df['Maneuver_2'].fillna('Missing')


In [105]:
df['plan']=df['plan'].astype(str)

In [106]:
df_dummies=pd.get_dummies(df[['Crossroad','Helmet','Point of impact','Gender','vehicle_type_2','Vehicle','Pavement','tram','Intersection'
                              , 'User category', 'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration', 'Age category',
                              'Accident location', 'Surface condition', 'Maneuver', 'Maneuver_2','Gender_2', 'Pedestrian localisation', 'Pedestrian action', 'Vehicle_2',
'Max speed', 'Long profile', 'Weather conditions', 'Road type', 'Trip purpose','Reflective jacket', 'plan','Point of impact_2', 'Obstacle','Gender_driver','Helmet_driver','age_driver','Year', 'Maneuver_3','Point of impact_3','Gender_3','Vehicle_3'

]])
df_dummies = df_dummies.astype(int)  # Convert boolean to integers



In [107]:
# Select columns that are not of type object
df_non_dummies = df[['age','severity','Number of passengers','number of involved vehicles','vma','largeurcha','pentemoyen','largeurtro','age_2','catu', 'Num_Acc','age_opposite_mean']]


In [108]:
df_non_dummies=pd.concat([df_non_dummies,df_dummies],axis=1)
df_non_dummies['road segment']=df_non_dummies['Crossroad_No intersection']

In [109]:
df_non_dummies=df_non_dummies.dropna(subset=['age','severity'])

In [110]:
df_non_dummies = df_non_dummies.reset_index(drop=True)

In [111]:
df_non_dummies['largeurcha'] = df_non_dummies['largeurcha'].fillna(999)
df_non_dummies['largeurtro'] = df_non_dummies['largeurtro'].fillna(999)
df_non_dummies['pentemoyen'] = df_non_dummies['pentemoyen'].fillna(999)
df_non_dummies['age_opposite_mean'] = df_non_dummies['age_opposite_mean'].fillna(999)

In [112]:
df_epmd=df_non_dummies.loc[df_non_dummies['Vehicle_E-scooter']==1]
df_ebike=df_non_dummies.loc[df_non_dummies['Vehicle_E-bike']==1]
df_bike=df_non_dummies.loc[df_non_dummies['Vehicle_Bike']==1]

In [113]:
df_carcrashes=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Cars']==1) | (df_non_dummies['vehicle_type_2_Large motorized vehicle'] ==1) | (df_non_dummies['vehicle_type_2_Light motorized vehicle']==1) ]
df_mmv=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Micromobility vehicle']==1) ]
df_alone=df_non_dummies.loc[df_non_dummies['vehicle_type_2_No other vehcile']==1]

df_carcrashes=df_carcrashes.loc[df_carcrashes['catu'].isin([1,2])]
df_mmv=df_mmv.loc[df_mmv['catu'].isin([1,2])]
df_mmv=df_mmv.loc[df_mmv['severity']!=3]
df_alone=df_alone.loc[df_alone['catu'].isin([1,2])]

num_acc_values = df_non_dummies.loc[df_non_dummies['vehicle_type_2_Pedestrian'] == 1, 'Num_Acc']

# Step 2: Filter the DataFrame to include rows with those Num_Acc values
df_pedestrian = df_non_dummies[df_non_dummies['Num_Acc'].isin(num_acc_values)]



df_carcrashes_epmd=df_epmd.loc[(df_epmd['vehicle_type_2_Cars']==1) | (df_epmd['vehicle_type_2_Large motorized vehicle'] ==1) | (df_epmd['vehicle_type_2_Light motorized vehicle']==1)]
df_carcrashes_ebike=df_ebike.loc[(df_ebike['vehicle_type_2_Cars']==1) | (df_ebike['vehicle_type_2_Large motorized vehicle'] ==1) | (df_ebike['vehicle_type_2_Light motorized vehicle']==1)]
df_carcrashes_bike=df_bike.loc[(df_bike['vehicle_type_2_Cars']==1) | (df_bike['vehicle_type_2_Large motorized vehicle'] ==1) | (df_bike['vehicle_type_2_Light motorized vehicle']==1)]


In [114]:
database_carcrashes = db.Database('database_carcrashes', df_carcrashes)
database_mmv= db.Database('database_mmv',df_mmv)
database_pedestrian= db.Database('database_pedestrian',df_pedestrian)
database_alone= db.Database('database_alone',df_alone)
database= db.Database('database',df_non_dummies)
database_carcrashes_epmd = db.Database('database_carcrashes_epmd', df_carcrashes_epmd)
database_carcrashes_ebike = db.Database('database_carcrashes_ebike', df_carcrashes_ebike)
database_carcrashes_bike = db.Database('database_carcrashes_bike', df_carcrashes_bike)


## Variables and Betas

In [115]:
import re
import biogeme.database as db
from biogeme.expressions import Beta, Variable


def normalize(name: str) -> str:
    # minuscules
    clean = name.lower()
    # remplacer tous les caractères non alphanumériques par '_'
    clean = re.sub(r'[^a-z0-9]+', '_', clean)
    # supprimer underscores multiples
    clean = re.sub(r'_+', '_', clean)
    # retirer _ au début/fin
    clean = clean.strip('_')
    # si commence par un chiffre → ajouter préfixe
    if re.match(r'^[0-9]', clean):
        clean = "var_" + clean
    return clean

base_suffixes = ['','_I', '_F', '_mean', '_I_mean', '_F_mean','_sd','_I_std','_F_std']
modes = ['', '_epmd', '_bike', '_ebike']  

for col in df_non_dummies.columns:
    clean_name = normalize(col)

    # Skip si nettoyage donne v_injuryde
    if clean_name == "":
        print(f"Skip (empty after cleaning): {col}")
        continue

    # Créer la Variable Biogeme si elle n'existe pas déjà
    if clean_name not in locals():
        # Variable garde le nom original entre quotes pour référence dans Biogeme
        exec(f"{clean_name} = Variable({repr(col)})")

    # Pour chaque suffixe de base et chaque mode, créer un Beta si non existant
    for suf in base_suffixes:
        for mode in modes:
            # Nom du beta : beta_<clean><suf><mode>  (mode peut être '' ou '_epmd' etc.)
            beta_var_name = f"beta_{clean_name}{suf}{mode}"
            # Nom qui sera donné à Biogeme (string)
            beta_label = beta_var_name

            # Choix des valeurs initiales : std souvent init à 1, les autres à 0
            if '_std' in suf or suf == '_std':
                init_value = 1
            else:
                init_value = 0

            # Si le nom de variable Python est invalide (rare ici) on skip
            if not beta_var_name.isidentifier():
                print(f"Skip beta (invalid identifier): {beta_var_name}")
                continue

            # Ne pas recréer si déjà présent
            if beta_var_name in locals():
                continue

            # Création v_injurya exec
            # signature Beta(name, initValue, lower, upper, status)
            # on laisse bornes None pour libre, status=0 (estimable)
            exec(f"{beta_var_name} = Beta({repr(beta_label)}, {init_value}, None, None, 0)")

constant_I = Beta('constant_I', 0, None, None, 0)
constant_F = Beta('constant_F', 0, None, None, 0)


## Model for car crashes

In [ ]:
v_no_injury = 0

v_injury = (  beta_gender_female_I * gender_female  + constant_I
      + beta_user_category_passenger_I * user_category_passenger
      + beta_number_of_involved_vehicles_I * number_of_involved_vehicles
      + beta_point_of_impact_back_I_bike * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
      + beta_point_of_impact_back_I_epmd* point_of_impact_back * vehicle_e_scooter
      + beta_vehicle_type_2_light_motorized_vehicle_I* vehicle_type_2_light_motorized_vehicle
      + beta_maneuver_2_overtaking_I * maneuver_2_overtaking
      + beta_maneuver_2_without_change_of_direction_I_bike * maneuver_without_change_of_direction * vehicle_bike
      + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
      + beta_intersection_no_intersection_I * intersection_no_intersection
)

v_fatality = (  constant_F
        + beta_user_category_passenger_I * user_category_passenger
        + beta_age_F * age
        + beta_vma_F * vma * intersection_no_intersection
        + beta_vehicle_type_2_large_motorized_vehicle_F * vehicle_type_2_large_motorized_vehicle
        +beta_vehicle_type_2_light_motorized_vehicle_F* vehicle_type_2_light_motorized_vehicle
        + beta_lighting_conditions_daylight_F * lighting_conditions_daylight
        + beta_maneuver_2_turning_left_F * maneuver_2_turning_left
        + beta_accident_location_on_cycle_facility_F* accident_location_on_cycle_facility
        + beta_lighting_conditions_night_with_street_lightings_on_F* lighting_conditions_night_with_street_lightings_on
        + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
)

utility_motorized_vehicles = {
    1: v_no_injury,
    2: v_injury,
    3: v_fatality
}


### Constant model

In [117]:

availability = {
    1: 1, 
    2: 1, 
    3: 1   
}

U={1:0,2:constant_I,3:constant_F}

model_name = 'InitialModel_carcrashes'


logprob = models.loglogit(U, availability, severity)

# Créez l'objet Biogeme
model_cst_car = bio.BIOGEME(database_carcrashes, logprob)
model_cst_car.modelName = model_name

# Estimation


results_constant_car = model_cst_car.estimate()



### Logit model

In [ ]:
# Random-parameters
sigma_I = Beta('sigma I', 0, None, None, 0)

X1 = bioDraws('X1', 'NORMAL')


# Adding the error component to the utilities
v_no_injury_rp=0
v_injury_rp = utility_motorized_vehicles[2] + sigma_I*X1
v_fatality_rp = utility_motorized_vehicles[3]

utility_motorized_vehicles_mixed={1:v_no_injury_rp,2:v_injury_rp,3:v_fatality_rp}

prob = models.logit(utility_motorized_vehicles_mixed,availability,severity)


# We integrate over B_TIME_RND using Monte-Carlo
logprob = log(MonteCarlo(prob))



# Create the Biogeme object
model_car  = bio.BIOGEME(database_carcrashes,logprob,number_of_draws=1)
model_car.modelName = "random_parameter_logit_car_crashes"

# Estimate the parameters. 
results_ml_carcrashes = model_car.estimate()


In [119]:
results_ml_carcrashes.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_accident_location_on_cycle_facility_F,-0.723108,0.502381,-1.439361,1.500483e-01
beta_age_F,0.036876,0.008799,4.190861,2.778982e-05
beta_gender_driver_female_I,-0.857892,0.418312,-2.050842,4.028236e-02
beta_gender_female_I,0.668299,0.137585,4.857362,1.189603e-06
beta_intersection_no_intersection_I,0.396965,0.143375,2.768722,5.627666e-03
beta_lighting_conditions_daylight_F,-1.045463,0.398011,-2.626717,8.621309e-03
beta_lighting_conditions_night_with_street_lightings_on_F,-0.312979,0.455467,-0.687161,4.919810e-01
beta_maneuver_2_overtaking_I,-0.382757,0.212698,-1.799531,7.193480e-02
beta_maneuver_2_turning_left_F,-0.995242,0.620822,-1.603102,1.089121e-01
beta_maneuver_2_without_change_of_direction_I_bike,0.508861,0.136338,3.732365,1.896904e-04


## MMV

In [ ]:
v_no_injury = 0

v_injury = (
      beta_gender_female_I                  * gender_female
    + constant_I
    + beta_gender_2_female_I               * (gender_2_female + gender_3_male)
    + beta_point_of_impact_back_I_bike     * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
    + beta_point_of_impact_back_I_epmd     * point_of_impact_back * vehicle_e_scooter
    + beta_surface_condition_wet_I         * surface_condition_wet
    + beta_age_I                           * age
    # + beta_maneuver_swerving_I           * maneuver_swerving
    + beta_maneuver_turning_left_I         * maneuver_turning_left
    + beta_age_2_I                          * age_2
    + beta_vehicle_2_e_scooter_I           * (vehicle_2_e_scooter * vehicle_3_e_scooter)
)

# Dictionnaire des fonctions d'utilité
utility_mmv = {
    1: v_no_injury,
    2: v_injury,
}


In [ ]:
availability1={1:1,2:1}

model_name = 'logit_mmv'

# Fichiers à écraser

logprob_2 = models.loglogit(utility_mmv, availability1, severity)
model_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_mmv.modelName = model_name
results_logit_mmv= model_mmv.estimate()
results_logit_mmv.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_2_I,-0.033680,0.004212,-7.995784e+00,1.332268e-15
beta_age_I,0.035462,0.005269,6.730291e+00,1.693246e-11
beta_gender_2_female_I,-0.955564,0.142891,-6.687354e+00,2.272404e-11
beta_gender_female_I,1.060571,0.170137,6.233634e+00,4.557366e-10
beta_maneuver_turning_left_I,-1.233114,0.314035,-3.926680e+00,8.612640e-05
beta_point_of_impact_back_I_bike,-1.070636,0.224493,-4.769132e+00,1.850217e-06
beta_point_of_impact_back_I_epmd,1.101805,0.677469,1.626353e+00,1.038746e-01
beta_surface_condition_wet_I,-0.620169,0.226798,-2.734454e+00,6.248377e-03
beta_vehicle_2_e_scooter_I,0.000000,0.000000,1.797693e+308,0.000000e+00
constant_I,0.852204,0.240200,3.547899e+00,3.883173e-04


In [123]:
v_no_injurym=0
v_injurym = constant_I


U={1:v_no_injurym,2:v_injurym}

In [124]:
model_name = 'InitialModel_mmv'



logprob_2 = models.loglogit(U, availability1, severity)

# Créez l'objet Biogeme
model_cst_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_cst_mmv.modelName = model_name

# Estimation


results_constant_mmv = model_cst_mmv.estimate()



## Pedestrian

In [125]:
Beta_age_bike_4=Beta('Beta_age_bike_4',0,None,None,0)

In [ ]:
utility_pedestrian = (
      beta_gender_female                     * gender_female
    + beta_age                               * age
    + beta_intersection_no_intersection      * intersection_no_intersection
    + beta_age_2                             * age_2
                                            * (number_of_involved_vehicles == 2)
    + beta_gender_2_female                   * gender_2_female
                                            * (number_of_involved_vehicles == 2)
    + beta_user_category_pedestrian          * user_category_pedestrian
    + beta_maneuver_2_turning_left           * user_category_pedestrian
                                            * (maneuver_2_turning_left + maneuver_2_turning_right)
                                            * (number_of_involved_vehicles == 2)
    + beta_crossroad_traffic_lights          * crossroad_traffic_lights
)


In [ ]:
model_name = 'ordered_probit_pedestrian'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_pedestrian,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_pedes.modelName = model_name
results_pedes = model_pedes.estimate()
results_pedes.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.014930,0.001612,9.260473,0.000000e+00
beta_age_2,-0.007405,0.001449,-5.111548,3.195302e-07
beta_crossroad_traffic_lights,0.227229,0.078058,2.911032,3.602367e-03
beta_gender_2_female,-0.568738,0.065118,-8.733942,0.000000e+00
beta_gender_female,0.516031,0.066935,7.709477,1.265654e-14
beta_intersection_no_intersection,0.244230,0.068624,3.558970,3.723115e-04
beta_maneuver_2_turning_left,0.529647,0.110500,4.793186,1.641537e-06
beta_user_category_pedestrian,1.346999,0.071884,18.738402,0.000000e+00
tau_1,0.791027,0.105418,7.503725,6.195044e-14
tau_1_diff_2,4.173075,0.170844,24.426164,0.000000e+00


In [128]:
model_name = 'ordinal_probit_pedestrian_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes.modelName = model_name
results_pedes_cst = model_cst_pedes.estimate()


## Single-vehicle

In [129]:
beta_user_category_passenger_mixed=beta_user_category_passenger_mean + beta_user_category_passenger_sd*X1 * bioDraws('X1', 'NORMAL')


In [130]:
utility_sv = (
    beta_age * age  +
    beta_user_category_passenger_mixed * user_category_passenger 
+ beta_long_profile_slope *long_profile_slope 
+beta_helmet_driver_yes_ebike*helmet_driver_yes*vehicle_e_bike
+ beta_number_of_passengers*user_category_driver*(number_of_passengers==1) 
)


In [131]:
model_name = 'ordered_probit_sinv'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log(MonteCarlo(the_chosen_proba))
model_solo_2 = bio.BIOGEME(database_alone, logprob,number_of_draws=1)
model_solo_2.modelName = model_name
results_solo_2 = model_solo_2.estimate()
results_solo_2.get_estimated_parameters()

The number of draws (1) is low. The results may not be meaningful.


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.008585,0.003626,2.367616,1.790309e-02
beta_helmet_driver_yes_ebike,1.782583,0.331964,5.369805,7.882191e-08
beta_long_profile_slope,0.321307,0.167613,1.916958,5.524332e-02
beta_number_of_passengers,-2.239493,0.232879,-9.616531,0.000000e+00
beta_user_category_passenger_mean,-1.887887,0.340919,-5.537646,3.065645e-08
beta_user_category_passenger_sd,-0.057448,0.103847,-0.553206,5.801226e-01
tau_1,-2.513135,0.192207,-13.075121,0.000000e+00
tau_1_diff_2,5.189950,0.196879,26.361088,0.000000e+00


In [132]:
model_name = 'ordered_logit_s_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_solo = bio.BIOGEME(database_alone, logprob)
model_cst_solo.modelName = model_name
results_cst_solo = model_cst_solo.estimate()
results_cst_solo.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
tau_1,-2.115696,0.064929,-32.584718,0.0
tau_1_diff_2,4.395508,0.099081,44.362708,0.0


## Likelihood-ratio test

In [133]:
def get_results(file_path):

    # Ouvrir le fichier en mode binaire
    with open(file_path, 'rb') as file:
        data = pickle.load(file)

    result = res.bioResults(data)
    
    # Retourner le résultat
    return result


#res_restricted=get_results('logit_mmv~51.pickle')
#res_unrestricted=get_results('panel_mmv~36.pickle')

#res_restricted.likelihood_ratio_test(res_unrestricted, 0.01)

## Out-of sample validation of the models

In [134]:
# Create DataFrames for each year and without each year
years = [2019, 2020, 2021, 2022, 2023]

# Classe pour contenir les données de validation
class ValidationData:
    def __init__(self, estimation, validation):
        self.estimation = estimation
        self.validation = validation

def create_validation_data(df):
    validation_data=[]
    validation_data.append(ValidationData(df[df['Year'].isin([2019, 2020, 2021, 2022])], df[df['Year'] == 2023]))
    df_lyon = df[df['Agglomeration_MÉTROPOLE DE LYON'] == 1]
    df_paris = df[df['Agglomeration_MÉTROPOLE DU GRAND PARIS']==1]
    validation_data.append(ValidationData(df_paris, df_lyon))
    return validation_data



# Create validation data for each dataset
validationData_car = create_validation_data(df_carcrashes)
validationData_mmv= create_validation_data(df_mmv)
validationData_alone_2 = create_validation_data(df_alone)
validationData_pedestrian = create_validation_data(df_pedestrian)

In [135]:
# Validate the model with the validation data for mmv
validation_results_car = model_car.validate(results_ml_carcrashes, validationData_car)
validation_results_car_cst = model_cst_car.validate(results_ml_carcrashes, validationData_car)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_car):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_car_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_car)):
    validation_loglike = validation_results_car[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_car_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on car (slide {i+1}): {rho_square}')





Log likelihood for 2129 validation data on car (slide 1): -349.3685372617828
Log likelihood for 1361 validation data on car (slide 2): -217.73020730388228
Log likelihood for 2129 validation data on car (constant model, slide 1): -400.5339948289545
Log likelihood for 1361 validation data on car (constant model, slide 2): -249.84096046680827
Rho-square for the validation data on car (slide 1): 0.12774310851946935
Rho-square for the validation data on car (slide 2): 0.12852477473241197


In [136]:
# Validate the model with the validation data for mmv
validation_results_mmv = model_mmv.validate(results_logit_mmv, validationData_mmv)
validation_results_mmv_cst = model_cst_mmv.validate(results_constant_mmv, validationData_mmv)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')





Log likelihood for 374 validation data on mmv (slide 1): -192.33769159646437
Log likelihood for 87 validation data on mmv (slide 2): -41.86523242443988
Log likelihood for 374 validation data on mmv (constant model, slide 1): -237.10399908707495
Log likelihood for 87 validation data on mmv (constant model, slide 2): -53.45166827834558
Rho-square for the validation data on mmv (slide 1): 0.18880452317537855
Rho-square for the validation data on mmv (slide 2): 0.216764718990064


In [137]:

# Validate the model with the validation data for mmv
validation_results_mmv = model_pedes.validate(results_pedes, validationData_pedestrian)
validation_results_mmv_cst = model_cst_pedes.validate(results_pedes_cst, validationData_pedestrian)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')




Log likelihood for 711 validation data on mmv (slide 1): -305.8780188215393
Log likelihood for 223 validation data on mmv (slide 2): -91.82829391930525
Log likelihood for 711 validation data on mmv (constant model, slide 1): -491.2867169242943
Log likelihood for 223 validation data on mmv (constant model, slide 2): -151.9994039479823
Rho-square for the validation data on mmv (slide 1): 0.3773940790899215
Rho-square for the validation data on mmv (slide 2): 0.39586411831765467


In [138]:


# Validate the model with the validation data for mmv
validation_results_solo_2 = model_solo_2.validate(results_solo_2, validationData_alone_2)
validation_results_solo_cst = model_cst_solo.validate(results_cst_solo, validationData_alone_2)

# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_solo_2):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_solo_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)

    # Calculate rho-square for each slide (mmv
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')


KeyboardInterrupt: 